# Lesson 6: Build a neural network

In Lesson 5 you watched a network learn. Now you build one yourself. A neural network in PyTorch is the two pieces from Lessons 3 and 4 wired together: linear layers and activations. Here you assemble them into one object, run data through it (the forward pass), turn the output into probabilities, and count the parameters. You will not train it yet, that is the next lesson.

Run every cell top to bottom, then do the exercises at the end and upload this notebook for feedback and a grade.

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## Way 1: nn.Sequential

`nn.Sequential` runs a list of layers one after another. This is the smallest real network: Linear, activation, Linear.

In [ ]:
model = nn.Sequential(
    nn.Linear(8, 16),   # input 8 -> hidden 16
    nn.ReLU(),          # activation
    nn.Linear(16, 3),   # hidden 16 -> output 3
)

x = torch.randn(4, 8)   # (batch=4, in=8)
y = model(x)
print(y.shape)          # torch.Size([4, 3])  (batch, out)

Calling `model(x)` runs the forward pass. The input's last axis (8) matches the first layer's `in_features`, and the output's last axis (3) is the last layer's `out_features`. The batch axis rides along untouched.

## Way 2: nn.Module (the real way)

Real models subclass `nn.Module`: define the layers in `__init__`, describe the forward pass in `forward`. This is the pattern you will use for every model in this course, including the GPT.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features, hidden, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_features)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

model = MLP(8, 16, 3)
print(model(torch.randn(4, 8)).shape)   # torch.Size([4, 3])
print(model)

Same network, but `forward` is plain Python now: you can print shapes inside it, add branches, reuse layers.

## Logits to probabilities: softmax

The raw outputs of the last linear layer are called **logits**: unbounded numbers, one per class. To read them as probabilities (positive, summing to 1), apply softmax over the last axis.

In [ ]:
logits = model(torch.randn(1, 8))   # shape (1, 3)
probs = F.softmax(logits, dim=-1)   # shape (1, 3)
print(probs)
print(probs.sum(dim=-1))            # tensor([1.])

The largest logit becomes the largest probability. During training you usually keep the logits and let the loss handle softmax internally (next lesson), but softmax is how you read a prediction.

## Counting parameters

Every weight and bias is a parameter that training will adjust. Count them with `p.numel()` over `model.parameters()`.

In [ ]:
total = sum(p.numel() for p in model.parameters())
print(total, 'parameters')

for name, p in model.named_parameters():
    print(name, tuple(p.shape))

For `MLP(8, 16, 3)`: layer 1 has 8x16 + 16 = 144, layer 2 has 16x3 + 3 = 51, total 195. The GPT you build later has millions, counted the exact same way.

## Exercises

Do these in the cells below, then upload the notebook. Only your code under each `# YOUR CODE HERE` line is graded.

In [ ]:
# E1: Build a network with nn.Sequential that maps 10 inputs -> 32 hidden -> ReLU -> 4 outputs.
#     Run a batch of shape (5, 10) through it and print the output shape (expect (5, 4)).
# YOUR CODE HERE (write your answer below)


In [ ]:
# E2: Write the same network as an nn.Module subclass called TwoLayerNet(in_features, hidden, out_features)
#     with an fc1, a ReLU, and an fc2. Instantiate TwoLayerNet(10, 32, 4), run a (5, 10) batch,
#     and print the output shape.
# YOUR CODE HERE (write your answer below)


In [ ]:
# E3: Turn logits into probabilities. Given logits = torch.tensor([[2.0, 1.0, 0.1]]),
#     apply softmax over the last dim and print the probabilities and their sum (expect 1.0).
# YOUR CODE HERE (write your answer below)


In [ ]:
# E4: Count the parameters of TwoLayerNet(10, 32, 4) with sum(p.numel() for p in model.parameters()).
#     Print the total (work it out by hand too: 10*32+32 + 32*4+4).
# YOUR CODE HERE (write your answer below)
